# quality

> feedback, a noise score, and a ranker fitted from both

In [ ]:
#| default_exp quality

Citation feedback, noise scoring and pairwise ranking are off by default. The noise score measured
0.988 AUC without labels and 0.996 after fitting. The pairwise ranker did not beat plain RRF
consistently (`evals/RESULTS.md`). Review `suggest_noisy` before calling `mark_noisy` or
`accept_noisy`. Keep the ranker off until it improves results on the target corpus.


In [ ]:
#| export
import json, math, re, time, uuid, warnings
import numpy as np
from collections import Counter, defaultdict
from fastcore.all import AttrDict, L, patch, first
from litesearch.quality import (NOISE_FEATURES, NOISE_W, Ranker, chunk_idf, noise_features,
                                noise_scores, _robust_z, _TOK)
from vishalakshi.core import Vault, DTYPE

## What was good

`SIGNALS` maps a signal to a label and a weight. A thumb is 3, a citation 1, `shown` is a 0.35-weight weak negative.


In [ ]:
#| export
#: signal -> (label, weight)
SIGNALS = dict(up=(1.0, 3.0), good=(1.0, 3.0), read=(1.0, 1.5), cited=(1.0, 1.0),
               shown=(0.0, 0.35), bad=(0.0, 3.0), down=(0.0, 3.0))

@patch
def _fb(self:Vault):
    'Feedback table.'
    return self.db.t.feedback

@patch
def rate(self:Vault,
         q:str,               # the question this judgement is about
         node_id:str=None,    # the section being judged
         doc_id:str=None,     # its document; derived from `node_id` when not given
         signal:str='up',     # one of SIGNALS
         rank:int=None,       # where it was ranked when it was shown
         score:float=None,    # the retrieval score it was shown with
         ask_id:str=None,     # groups the rows from one question together
) -> dict:
    'Record one section judgement.'
    if signal not in SIGNALS: raise ValueError(f'not a signal: {signal!r}; expected {sorted(SIGNALS)}')
    label, weight = SIGNALS[signal]
    row = dict(id=uuid.uuid4().hex[:16], at=time.time(), store=self.name, q=q or '',
               ask_id=ask_id or uuid.uuid4().hex[:12], node_id=node_id,
               doc_id=doc_id or (node_id.split('#')[0] if node_id else None),
               rank=rank, score=score, signal=signal, label=label, weight=weight)
    self._fb().insert(row)
    return row

@patch
def ratings(self:Vault, doc_id:str=None, limit:int=None) -> L:
    'Feedback for this shelf, newest first.'
    w = f'store={self.name!r}' + (f' AND doc_id={doc_id!r}' if doc_id else '')
    return L(self._fb()(where=w, order_by='at desc', limit=limit))


In [ ]:
import tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as d:
    v = Vault(Path(d)/'test.db', offline=True)

    assert 'feedback' in v.db.t
    assert v.ratings() == []

    v.rate('question', node_id='doc#1', signal='up')
    assert len(v.ratings()) == 1

### Citations are the label

`ask` already knows which sections it cited. `learn()` logs that; those rows are the supervision for noise and for the ranker.


In [ ]:
#| export
@patch
def log_ask(self:Vault,
            out,             # what `ask` returned
            up:list=None,    # section numbers you thought were good, as `[1]`-style numbers
            down:list=None,  # section numbers you thought were bad
            weak:bool=True,  # log the uncited sections as weak negatives
) -> int:
    """Write feedback rows for one answered question. Prefer `learn()` when you have `ask`'s `out`."""
    up, down = set(up or ()), set(down or ())
    res = L((out.get('context') or {}).get('results') or ())
    cited = {c['n'] for c in (out.get('cited') or L())}
    if not (cited or up or down): return 0     # see above: no citations is not six negatives
    q, aid = out.get('question') or '', uuid.uuid4().hex[:12]
    n = 0
    for i, r in enumerate(res, 1):
        sig = ('up' if i in up else 'down' if i in down else 'cited' if i in cited else 'shown' if weak else None)
        if sig is None or not getattr(r, 'node_id', None): continue
        self.rate(q, node_id=r.node_id, doc_id=getattr(r, 'doc_id', None), signal=sig, rank=i,
                  score=float(getattr(r, 'score', 0.0) or 0.0), ask_id=aid)
        n += 1
    return n

@patch
def _observe(self:Vault, out):
    "The hook `ask` calls on its way out; silent unless `v.learn()` turned logging on."
    if getattr(self, '_learning', None) is None:
        try: self._learning = bool((first(self._rankers()(where=f'store={self.name!r}')) or {}).get('logging'))
        except Exception: self._learning = False
    if self._learning:
        try: self.log_ask(out)
        except Exception as e: warnings.warn(f'feedback not logged ({type(e).__name__}: {e})')
    return out


## The Beta prior

Per-document usefulness is a Beta updated from citation hits and misses. Weak evidence stays near the prior; a cookie banner cited never sinks fast.


In [ ]:
#| export
@patch
def doc_stats(self:Vault, exclude_ask=None) -> dict:
    """doc_id -> (weighted wins, weighted losses) over the feedback log for this shelf."""
    ex = set(exclude_ask or ())
    out = defaultdict(lambda: [0.0, 0.0])
    for r in self._fb()(where=f'store={self.name!r}'):
        if not r['doc_id'] or r['ask_id'] in ex: continue
        w = float(r['weight'] or 1.0)
        out[r['doc_id']][0 if (r['label'] or 0) >= 0.5 else 1] += w
    return {k: tuple(v) for k, v in out.items()}

@patch
def doc_prior(self:Vault,
              gamma:float=0.35,   # how hard the prior is allowed to push; 0 disables it
              explore:float=1.0,  # 1 -> Thompson sample the posterior, 0 -> use its mean
              lo:float=0.6, hi:float=1.6,   # clamp, so no amount of evidence can bury a document
              seed:int=None,
              exclude_ask=None,   # ignore these questions' rows; for an out-of-fold prior
) -> dict:
    "doc_id -> a multiplier on its retrieval score, from the Beta posterior over its feedback."
    rng, out = np.random.default_rng(seed), {}
    for did, (wins, losses) in self.doc_stats(exclude_ask=exclude_ask).items():
        a, b = 1.0 + wins, 1.0 + losses
        p = rng.beta(a, b) if (explore and rng.random() < explore) else a/(a+b)
        out[did] = float(np.clip((p/0.5)**gamma, lo, hi))
    return out

## Noise candidates

`suggest_noisy` ranks junk candidates from the log and document features. `mark_noisy` is the durable exclusion from `search` / `sections` / `context` / `ask`.


The features, the arithmetic behind them and the `Ranker` are
[litesearch's](https://Karthik777.github.io/litesearch/quality.html): they read a chunk store and
nothing else, and the 0.988 AUC is measured there. What is on this page is the judgement over the
top of them, which documents a person marked, and whether the score is allowed to exclude
anything.

In [ ]:
#| export
@patch
def _seen(self:Vault) -> dict:
    "doc_id -> the questions it was retrieved for, from the feedback log. The `promiscuity` feature."
    out = {}
    try:
        for r in self._fb()(where=f'store={self.name!r}'):
            if r['doc_id']: out.setdefault(r['doc_id'], set()).add(r['q'])
    except Exception: pass
    return out

@patch
def noise_features(self:Vault, **kw) -> AttrDict:
    "litesearch's features over this shelf, with the vault's own retrieval log behind `promiscuity`."
    return noise_features(self.db, store=self.name, seen=self._seen(), dtype=DTYPE, **kw)

@patch
def noise_scores(self:Vault, weights:dict=None, ranker=None, **kw) -> L:
    'Every document, most suspicious first, with the features that put it there.'
    return noise_scores(self.db, store=self.name, weights=weights, ranker=ranker,
                        seen=self._seen(), dtype=DTYPE, **kw)

@patch
def chunk_matrix(self:Vault, limit:int=None, seed:int=0) -> tuple:
    "`(ids, doc_ids, texts, V)` for this shelf: litesearch holds the vectors, this names the store."
    return self.db.chunk_matrix(store=self.name, limit=limit, seed=seed, dtype=DTYPE)

In [ ]:
#| export
@patch
def suggest_noisy(self:Vault, k:int=20, min_score:float=1.0, **kw) -> L:
    "The `k` documents most worth looking at, excluding ones already judged for noise."
    # only `noisy` judgments count: a `mark_not_pii` row must not hide a footer from review
    judged = {r['doc_id'] for r in self._marks()(where=f'store={self.name!r} AND noisy IS NOT NULL')}
    return self.noise_scores(ranker=self._noise_rk(), **kw).filter(
        lambda r: r.score >= min_score and r.doc_id not in judged)[:k]

@patch
def mark_noisy_many(self:Vault, refs, noisy:bool=True, reason:str='') -> L:
    "Mark many documents at once: the accept step after `suggest_noisy`."
    return L(refs).map(lambda r: self.mark_noisy(getattr(r, 'doc_id', r), noisy=noisy, reason=reason))

@patch
def accept_noisy(self:Vault, k:int=20, reason:str='suggested', **kw) -> L:
    "Mark the current top noise suggestions. Suggestions stay suggestions until this (or `mark_noisy`)."
    return self.mark_noisy_many(self.suggest_noisy(k=k, **kw), reason=reason)

### Nearest neighbours

Near a known-noisy document is a useful signal. The scorer mixes that with citation and surface features; numbers in `evals/RESULTS.md`.


## The ranker

Pairwise linear LTR over the ask log. `fit_ranker(save=True)` writes weights; `use_ranker(True)` is a separate call, deliberately. Off by default because no learned reranker beat plain RRF reproducibly across the measured cells.


### The features

Features are cheap document- and query-side signals already available at retrieve time. See the feature list in code; methodology stays in `evals/`.


In [ ]:
#| export
def _epoch(v) -> float:
    "`added_at` is an epoch on some rows and a SQL timestamp string on others. Take either."
    if v is None: return time.time()
    try: return float(v)
    except (TypeError, ValueError): pass
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%dT%H:%M:%S'):
        try: return time.mktime(time.strptime(str(v)[:26], fmt))
        except ValueError: continue
    return time.time()

PAIR_FEATURES = ('score', 'rank', 'gap', 'overlap', 'idf_overlap', 'cos', 'len',
                 'prior', 'is_code', 'is_web', 'is_note', 'age') + NOISE_FEATURES

@patch
def _doc_side(self:Vault, refresh:bool=False) -> tuple:
    "Cached per-document features: the noise block, the Beta prior, kind and age."
    if refresh or getattr(self, '_ds', None) is None:
        f = self.noise_features()
        nz = dict(zip(f.doc_ids, _robust_z(f.X))) if len(f.doc_ids) else {}
        meta = {r['id']: (r['kind'] or '', _epoch(r['added_at'])) for r in self.t.docs(select='id, kind, added_at')}
        self._ds = (nz, meta)
    return self._ds

@patch
def pair_X(self:Vault, q:str, hits, prior:dict=None) -> np.ndarray:
    "The design matrix for one question's hits, in `PAIR_FEATURES` order."
    nz, meta = self._doc_side()
    prior = self.doc_prior() if prior is None else prior
    idf = getattr(self, '_idf_cache', None)
    if idf is None: idf = self._idf_cache = chunk_idf(self.chunk_matrix(limit=20_000)[2])
    qt = set(_TOK.findall((q or '').lower()))
    qw = sum(idf.get(w, 0.0) for w in qt) or 1.0
    qv = np.frombuffer(self.qemb(q), dtype=DTYPE).astype(np.float32) if q else None
    if qv is not None: qv /= np.linalg.norm(qv) + 1e-9
    top = max((float(getattr(h, 'score', 0.0) or 0.0) for h in hits), default=0.0) or 1.0
    now, rows = time.time(), []
    for i, h in enumerate(hits, 1):
        txt = str(getattr(h, 'text', '') or '')
        ht = set(_TOK.findall(txt.lower()))
        sc = float(getattr(h, 'score', 0.0) or 0.0)
        did = getattr(h, 'doc_id', None)
        kind, added = meta.get(did, ('', now))
        cos = 0.0
        if qv is not None and getattr(h, 'node_id', None):
            vs = [np.frombuffer(r['embedding'], dtype=DTYPE).astype(np.float32)
                  for r in self.store(where=f"node_id={h.node_id!r}", select='embedding') if r['embedding']]
            if vs:
                m = np.mean(vs, 0); cos = float(qv @ (m / (np.linalg.norm(m) + 1e-9)))
        rows.append([sc, 1.0/i, sc/top,
                     len(qt & ht)/max(len(qt), 1),
                     sum(idf.get(w, 0.0) for w in qt & ht)/qw,
                     cos, math.log1p(len(txt)),
                     float(prior.get(did, 1.0)),
                     float(kind == 'code'), float(kind == 'web'), float(kind == 'note'),
                     math.log1p(max(now - added, 0)/86400.0)]
                    + list(nz.get(did, np.zeros(len(NOISE_FEATURES)))))
    return np.array(rows, np.float64) if rows else np.zeros((0, len(PAIR_FEATURES)))

## Fitting, storing, switching on

Fit, inspect, and enable in separate calls. Whether a ranker belongs on is a question for `evals/`.


In [ ]:
#| export
@patch
def learn(self:Vault, on:bool=True) -> dict:
    "Log every `ask` as feedback from now on. Off by default: nothing is recorded until you say so."
    t = self._rankers()
    row = dict(first(t(where=f'store={self.name!r}')) or dict(store=self.name, model='', at=time.time(),
                                                              enabled=0, note=''))
    row.update(logging=int(bool(on)), store=self.name)
    t.insert(row, replace=True); self._learning = bool(on)
    return row

@patch
def training_data(self:Vault, min_group:int=2) -> AttrDict:
    """Feedback as `(X, groups, y, weights)`. Drops all-positive groups (pairwise LTR learns nothing)."""
    by = defaultdict(list)
    for r in self._fb()(where=f'store={self.name!r}', order_by='at'): by[r['ask_id']].append(r)
    Xs, gs, ys, ws = [], [], [], []
    nodes = {}
    for aid, rows in by.items():
        labs = {float(r['label'] or 0) for r in rows}
        if len(rows) < min_group or len(labs) < 2: continue
        hits = L()
        for r in rows:
            if r['node_id'] not in nodes:
                nodes[r['node_id']] = (self.read(r['node_id'], max_chars=4000) or {}).get('text', '')
            hits.append(AttrDict(node_id=r['node_id'], doc_id=r['doc_id'], text=nodes[r['node_id']],
                                 score=float(r['score'] or 0.0)))
        X = self.pair_X(rows[0]['q'], hits, prior=self.doc_prior(explore=0.0, exclude_ask={aid}))
        if not len(X): continue
        Xs.append(X); gs += [aid]*len(X)
        ys += [float(r['label'] or 0) for r in rows]; ws += [float(r['weight'] or 1.0) for r in rows]
    if not Xs: return AttrDict(X=np.zeros((0, len(PAIR_FEATURES))), groups=[], y=[], weights=[],
                               names=list(PAIR_FEATURES))
    return AttrDict(X=np.vstack(Xs), groups=gs, y=np.array(ys), weights=np.array(ws),
                    names=list(PAIR_FEATURES))

@patch
def fit_ranker(self:Vault, l2:float=1.0, save:bool=False, **kw) -> Ranker:
    "Fit a `Ranker` on everything logged so far. `save=True` stores it; it is still not switched on."
    d = self.training_data()
    r = Ranker(d.names).fit(d.X, d.groups, d.y, weights=d.weights, l2=l2, **kw)
    if save and r.w is not None:
        t = self._rankers()
        row = dict(first(t(where=f'store={self.name!r}')) or dict(store=self.name, enabled=0, logging=0, note=''))
        row.update(store=self.name, model=json.dumps(r.to_dict()), at=time.time())
        t.insert(row, replace=True)
        self._rk_cache = None
    return r

@patch
def fit_noise(self:Vault, labels:dict=None, l2:float=1.0, save:bool=False, **kw) -> Ranker:
    """Fit noise weights from marked docs. Does not enable scoring; call `use_noise(True)`."""
    labels = labels or {r['doc_id']: bool(r['noisy']) for r in self._marks()(where=f'store={self.name!r}')
                        if r['noisy'] is not None}
    if not labels: raise ValueError('nothing marked: mark_noisy a few documents first')
    f = self.noise_features(**kw)
    keep = [i for i, d in enumerate(f.doc_ids) if d in labels] or list(range(len(f.doc_ids)))
    y = np.array([float(labels.get(f.doc_ids[i], 0.0)) for i in range(len(f.doc_ids))])
    if len(set(y[keep].tolist())) < 2:
        keep = list(range(len(f.doc_ids)))   # only positives marked: the unmarked ones are the negatives
    X = _robust_z(f.X)[keep]
    r = Ranker(f.names).fit(X, ['all']*len(keep), y[keep], l2=l2)
    if save and r.w is not None:
        t = self._rankers()
        row = dict(first(t(where=f'store={self.name!r}')) or dict(store=self.name, model='', enabled=0,
                                                                  logging=0, note='', noise_on=0))
        row.update(store=self.name, noise=json.dumps(r.to_dict()), at=time.time())
        t.insert(row, replace=True)
        self._noise_cache = None
    return r

@patch
def use_noise(self:Vault, on:bool=True) -> dict:
    "Put a stored noise blend behind `suggest_noisy`. Separate from fitting it."
    t = self._rankers()
    row = first(t(where=f'store={self.name!r}'))
    if not row or not row.get('noise'): raise ValueError('no noise blend stored; fit_noise(save=True) first')
    row = dict(row, noise_on=int(bool(on)))
    t.insert(row, replace=True); self._noise_cache = None
    return dict(store=self.name, noise_on=bool(on), fitted_at=row['at'])

@patch
def _noise_rk(self:Vault):
    "The enabled noise blend for this shelf, or None."
    if getattr(self, '_noise_cache', None) is None:
        try:
            r = first(self._rankers()(where=f'store={self.name!r} AND noise_on=1'))
            self._noise_cache = Ranker.from_dict(json.loads(r['noise'])) if (r and r.get('noise')) else False
        except Exception: self._noise_cache = False
    return self._noise_cache or None

@patch
def use_ranker(self:Vault, on:bool=True) -> dict:
    "Put the stored ranker in the path of `context`, and so of `ask`. Separate from fitting it."
    t = self._rankers()
    row = first(t(where=f'store={self.name!r}'))
    if not row or not row['model']: raise ValueError('no ranker stored; fit_ranker(save=True) first')
    row = dict(row, enabled=int(bool(on)))
    t.insert(row, replace=True); self._rk_cache = None
    return dict(store=self.name, enabled=bool(on), fitted_at=row['at'])

@patch
def _rk(self:Vault):
    "The enabled ranker for this shelf, or None. Read once per Vault, not once per question."
    if getattr(self, '_rk_cache', None) is None:
        try:
            r = first(self._rankers()(where=f'store={self.name!r} AND enabled=1'))
            self._rk_cache = Ranker.from_dict(json.loads(r['model'])) if (r and r['model']) else False
        except Exception: self._rk_cache = False
    return self._rk_cache or None

@patch
def retune(self:Vault, q:str, hits, ranker=None, alpha:float=0.25, rrf_k:int=60) -> L:
    """Reorder hits fused with baseline RRF (`alpha=0.25`: -0.001 ΔnDCG@10 vs -0.127 substitute)."""
    rk = ranker or self._rk()
    if rk is None or not len(hits): return L(hits)
    s = rk.score(self.pair_X(q, hits))
    order = sorted(range(len(hits)), key=lambda i: -s[i])
    lrank = {i: r for r, i in enumerate(order)}
    for i, h in enumerate(hits):
        h.tuned = float(s[i])
        h.fused = 1.0/(rrf_k + i) + alpha/(rrf_k + lrank[i])
    return L(sorted(hits, key=lambda h: -h.fused))

@patch
def _post(self:Vault, q:str, ctx):
    "The hook `context` calls: a no-op until a ranker has been fitted *and* switched on."
    if self._rk() is None: return ctx
    try: ctx.results, ctx.tuned = self.retune(q, ctx.results), True
    except Exception as e: warnings.warn(f'ranker not applied ({type(e).__name__}: {e})')
    return ctx


## A worked check

Synthetic shape check only; retrieval results are in `evals/`.


In [ ]:
#| hide
import tempfile
from pathlib import Path
from fastcore.test import test_eq

v = Vault(Path(tempfile.mkdtemp())/'q.db', offline=True, dims=64)
topics = [
    'Late chunking keeps document context around each passage for retrieval.',
    'Reciprocal rank fusion merges ordered lists without shared vector spaces.',
    'Entity graphs connect people and organisations mentioned across notes.',
    'Static encoders trade a little recall for orders of magnitude faster indexing.',
    'A shelf partitions one vault file so two corpora stop diluting each other.',
    'Structured extraction pulls invoice totals into fields a table can hold.',
]
for i, t in enumerate(topics):
    v.add(t + f' Further detail for document {i} about the topic above.',
          title=f'Real {i}', source=f'real{i}')
boiler = ('We use cookies and similar technologies to personalise content. '
          'All rights reserved. Contact us. Privacy policy. Terms of service.')
for i in range(4):
    v.add(boiler, title=f'Boiler {i}', source=f'b{i}')
f = v.noise_features()
assert f.X.shape == (10, len(NOISE_FEATURES)), f.X.shape
sc = v.noise_scores()
assert len(sc) == 10 and sc[0].score >= sc[-1].score

# suggest_noisy ignores pii-only marks
v.mark_not_pii('real0', reason='test')
ids = set(v.suggest_noisy(k=20, min_score=-10).attrgot('doc_id'))
test_eq(v.doc('real0')['id'] in ids, True)

# hard exclusion from retrieval
v.mark_noisy('b0', reason='footer')
test_eq(any('Boiler 0' in (h.get('breadcrumb') or '') for h in v.search('cookies')), False)

# batch accept, and a fitted blend round-trips
v.mark_noisy('b1', reason='footer')
v.mark_noisy('real1', noisy=False, reason='keep')
rk = v.fit_noise(save=True)
assert rk.w is not None
v.use_noise(True)
got = v.accept_noisy(k=1, min_score=-10, reason='batch')
assert len(got) >= 1
print('noise plumbing ok', [(r.title, round(r.score, 2)) for r in sc[:3]])

noise plumbing ok [('Real 3', 0.5), ('Real 2', 0.27), ('Real 4', 0.12)]


In [ ]:
#| hide
# a fabricated feedback log, only to show the fit runs and the weights come out readable
import uuid as _uuid
for qi, q in enumerate(['reciprocal rank fusion', 'chunk size', 'evaluation mrr']):
    hits = v.sections(q, limit=6)
    aid = _uuid.uuid4().hex[:12]
    for i, s in enumerate(hits, 1):
        good = 'Boiler' not in (s['breadcrumb'] or '')
        v.rate(q, node_id=s['node_id'], signal='cited' if good else 'shown', rank=i,
               score=float(s['score']), ask_id=aid)
d = v.training_data()
print('training rows', d.X.shape, 'groups', len(set(d.groups)))
r = v.fit_ranker()
print(r)
assert r.w is not None and len(r.w) == len(PAIR_FEATURES)
print(r.weights()[:4])

training rows (18, 21) groups 3
Ranker(prior=+0.19, low_idf=-0.15, off_centre=+0.13, len=-0.09, spread_chunk=+0.08, ...)
[{'feature': 'prior', 'weight': 0.1875}, {'feature': 'low_idf', 'weight': -0.1479}, {'feature': 'off_centre', 'weight': 0.1312}, {'feature': 'len', 'weight': -0.0933}]
